In [20]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import scipy.sparse as sp
import joblib
import pandas as pd
import time
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import brier_score_loss,average_precision_score
from sklearn.isotonic import IsotonicRegression

In [ ]:
print("1. Loading frozen artifacts...")

X_final_test = sp.load_npz('../data/05_fused/X_final_test.npz')
y_final_test = np.load('../data/05_fused/y_final_test.npy')

xgb_fused = joblib.load("../models/xgb_fused_model.joblib")
calibrated_xgb = joblib.load("../models/calibrated_xgb_model.joblib")

print("2. Generating predictions from loaded models...")


y_prob_uncaliberated = xgb_fused.predict_proba(X_final_test)[:,2] 
y_prob_caliberated = calibrated_xgb.predict_proba(X_final_test)[:,2] 

y_final_test_binary = (y_final_test == 2).astype(int)

print("\n--- 3. Calibration Evaluation (Brier Score) ---")

brier_uncal = brier_score_loss(y_final_test_binary, y_prob_uncaliberated)
brier_cal = brier_score_loss(y_final_test_binary, y_prob_caliberated)
print(f"Uncalibrated XGBoost Brier Score: {brier_uncal:.4f}")
print(f"Isotonic Calibrated Brier Score:  {brier_cal:.4f}")

print("\n--- 4. SOC Triage Evaluation (Recall@K) & (Precision@K) ---")

traige_queue = pd.DataFrame({
    'true_label' : y_final_test_binary,
    'caliberated_prob' : y_prob_caliberated
})

traige_queue = traige_queue.sort_values(by='caliberated_prob',ascending=False)
total_actual_threats = traige_queue['true_label'].sum()

print(f"Total Actual Threats in Final Test Set: {total_actual_threats}\n")

for k in [100, 500, 1000, 2500,9000]:
    top_k = traige_queue.head(k)
    threats_caught = top_k['true_label'].sum()
    recall_at_k = threats_caught/total_actual_threats
    precision_at_k = threats_caught/k
    print(f"Recall@{k:<4}: {recall_at_k:>7.2%}  (Caught {threats_caught} out of {total_actual_threats} threats)")
    print(f"Precison@{k:<4}: {precision_at_k:>7.2%}  ({threats_caught} threats were there in {k} evidences)")

1. Loading frozen artifacts...
2. Generating predictions from loaded models...

--- 3. Calibration Evaluation (Brier Score) ---
Uncalibrated XGBoost Brier Score: 0.1189
Isotonic Calibrated Brier Score:  0.1090

--- 4. SOC Triage Evaluation (Recall@K) & (Precision@K) ---
Total Actual Threats in Final Test Set: 9541

Recall@100 :   1.05%  (Caught 100 out of 9541 threats)
Precison@100 : 100.00%  (100 threats were there in 100 evidences)
Recall@500 :   5.24%  (Caught 500 out of 9541 threats)
Precison@500 : 100.00%  (500 threats were there in 500 evidences)
Recall@1000:  10.48%  (Caught 1000 out of 9541 threats)
Precison@1000: 100.00%  (1000 threats were there in 1000 evidences)
Recall@2500:  24.90%  (Caught 2376 out of 9541 threats)
Precison@2500:  95.04%  (2376 threats were there in 2500 evidences)
Recall@9000:  57.65%  (Caught 5500 out of 9541 threats)
Precison@9000:  61.11%  (5500 threats were there in 9000 evidences)


In [3]:
print(len(y_final_test))
print(pd.Series(y_final_test_binary).value_counts())
print(pd.Series(y_final_test_binary).value_counts(normalize=True))

44785
0    35244
1     9541
Name: count, dtype: int64
0    0.78696
1    0.21304
Name: proportion, dtype: float64


In [4]:
print("--- Global Ranking Evaluation (PR-AUC) ---")
pr_auc_uncal = average_precision_score(y_final_test_binary,y_prob_uncaliberated)
pr_auc_cal = average_precision_score(y_final_test_binary,y_prob_caliberated)

print(f"Uncalibrated PR-AUC: {pr_auc_uncal:.4f}")
print(f"Calibrated PR-AUC:   {pr_auc_cal:.4f}")

print("\n--- Inference Latency ---")
start_time = time.time()
xgb_fused.predict_proba(X_final_test)
xgb_time = time.time() - start_time

print(f"XGBoost Inference Time (Total Test Set): {xgb_time:.4f} seconds")

--- Global Ranking Evaluation (PR-AUC) ---
Uncalibrated PR-AUC: 0.6860
Calibrated PR-AUC:   0.6898

--- Inference Latency ---
XGBoost Inference Time (Total Test Set): 0.1046 seconds


## Building a MLP

In [10]:
print("1. Loading Training Data...")

X_train = sp.load_npz("../data/05_fused/X_train_fused.npz")
y_train = np.load("../data/05_fused/y_train.npy")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("\n2. Building the Sparse DataLoader...")

class SparseDataset(Dataset):
    def __init__(self,X_sparse,y_numpy):
        self.X = X_sparse
        self.y = y_numpy

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self,idx):
        x_dense = torch.tensor(self.X[idx].toarray().squeeze(),dtype=torch.float32)
        y_tensor = torch.tensor(self.y[idx],dtype = torch.long)
        return x_dense,y_tensor

batch_size = 512
input_dim = X_train.shape[1]

train_dataset = SparseDataset(X_train,y_train)
train_loader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True)

print("\n3. Defining the Neural Network...")

class TriageMLP(nn.Module):
    def __init__(self,input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim,128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64,3)
        )
    def forward(self,x):
        return self.network(x)

model = TriageMLP(input_dim).to(device)
print(model)
        

1. Loading Training Data...
Using device: cuda

2. Building the Sparse DataLoader...

3. Defining the Neural Network...
TriageMLP(
  (network): Sequential(
    (0): Linear(in_features=255, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=3, bias=True)
  )
)


In [11]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

epochs = 3
print("Starting GPU Training...")
model.train()
for epoch in range(epochs):
    start_time = time.time()
    total_loss = 0
    for batch_X,batch_y in train_loader:
        batch_X,batch_y = batch_X.to(device),batch_y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(batch_X),batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Time: {time.time() - start_time:.2f}s")

Starting GPU Training...
Epoch 1/3 | Loss: 33.3678 | Time: 22.28s
Epoch 2/3 | Loss: 0.7958 | Time: 22.11s
Epoch 3/3 | Loss: 0.7450 | Time: 22.04s


In [18]:
print("\nEvaluating MLP against Test Set...")
model.eval()
X_test_tensor = torch.tensor(X_final_test.toarray(),dtype=torch.float32).to(device)
start_time = time.time()

with torch.no_grad():
    logits = model(X_test_tensor)
    mlp_probs_class_2 = torch.softmax(logits,dim=1).cpu().numpy()[:,2]
mlp_inference_time = time.time() - start_time

mlp_brier = brier_score_loss(y_final_test_binary,mlp_probs_class_2)
mlp_prauc = average_precision_score(y_final_test_binary,mlp_probs_class_2)

triage_queue = pd.DataFrame({
    'true_label' : y_final_test_binary,
    'prob' : mlp_probs_class_2
}).sort_values(by='prob',ascending=False)

total_actual_threats = traige_queue['true_label'].sum()

print(f"\n--- EXPERIMENT RESULTS: UNCALIBRATED MLP vs XGBOOST ---")
print(f"Latency (Test Set):  XGB: 0.1162s  |  MLP: {mlp_inference_time:.4f}s")
print(f"Brier Score:         XGB: 0.1189   |  MLP: {mlp_brier:.4f}")
print(f"PR-AUC (Global):     XGB: 0.6860   |  MLP: {mlp_prauc:.4f}")

print("\n --- Precison@k and Recall @k ---")
for k in [100,500,1000,2500,9000]:
    top_k = traige_queue.head(k)
    threats_caught = top_k['true_label'].sum()
    recall_at_k = threats_caught/total_actual_threats
    precision_at_k = threats_caught/k
    print(f"Recall@{k:<4}: {recall_at_k:>7.2%}  (Caught {threats_caught} out of {total_actual_threats} threats)")
    print(f"Precison@{k:<4}: {precision_at_k:>7.2%}  ({threats_caught} threats were there in {k} evidences)")


Evaluating MLP against Test Set...

--- EXPERIMENT RESULTS: UNCALIBRATED MLP vs XGBOOST ---
Latency (Test Set):  XGB: 0.1162s  |  MLP: 0.0277s
Brier Score:         XGB: 0.1189   |  MLP: 0.1214
PR-AUC (Global):     XGB: 0.6860   |  MLP: 0.5842

 --- Precison@k and Recall @k ---
Recall@100 :   1.05%  (Caught 100 out of 9541 threats)
Precison@100 : 100.00%  (100 threats were there in 100 evidences)
Recall@500 :   5.24%  (Caught 500 out of 9541 threats)
Precison@500 : 100.00%  (500 threats were there in 500 evidences)
Recall@1000:  10.48%  (Caught 1000 out of 9541 threats)
Precison@1000: 100.00%  (1000 threats were there in 1000 evidences)
Recall@2500:  24.06%  (Caught 2296 out of 9541 threats)
Precison@2500:  91.84%  (2296 threats were there in 2500 evidences)
Recall@9000:  50.61%  (Caught 4829 out of 9541 threats)
Precison@9000:  53.66%  (4829 threats were there in 9000 evidences)


In [22]:
print("Generating MLP predictions on Calibration Set...")
X_calib = sp.load_npz("../data/05_fused/X_calib.npz")
y_calib = np.load("../data/05_fused/y_calib.npy")
y_calib_binary = (y_calib == 2).astype(int)

X_calib_tensor = torch.tensor(X_calib.toarray(),dtype=torch.float32).to(device)
model.eval()
with torch.no_grad():
    calib_logits = model(X_calib_tensor)
    mlp_calib_probs = torch.softmax(calib_logits,dim=1).cpu().numpy()[:,2]

print("Fitting Isotonic Calibrator on MLP...")
mlp_caliberator = IsotonicRegression(out_of_bounds='clip')
mlp_caliberator.fit(mlp_calib_probs,y_calib_binary)

mlp_calib_probs = mlp_caliberator.transform(mlp_probs_class_2)

mlp_brier_cal = brier_score_loss(y_final_test_binary,mlp_calib_probs)
mlp_prauc_cal = average_precision_score(y_final_test_binary,mlp_calib_probs)

print("\n=======================================================")
print("             FINAL HEAD-TO-HEAD BENCHMARK              ")
print("=======================================================")
print(f"{'Metric':<20} | {'XGBoost (Calibrated)':<20} | {'MLP (Calibrated)':<20}")
print("-" * 67)
print(f"{'Brier Score':<20} | {brier_cal:<20.4f} | {mlp_brier_cal:<20.4f}")
print(f"{'PR-AUC (Global)':<20} | {pr_auc_cal:<20.4f} | {mlp_prauc_cal:<20.4f}")
print(f"{'Inference Time':<20} | {'0.1162s (CPU)':<20} | {mlp_inference_time:<14.4f}s (GPU)")
print("=======================================================")

Generating MLP predictions on Calibration Set...
Fitting Isotonic Calibrator on MLP...

             FINAL HEAD-TO-HEAD BENCHMARK              
Metric               | XGBoost (Calibrated) | MLP (Calibrated)    
-------------------------------------------------------------------
Brier Score          | 0.1090               | 0.1208              
PR-AUC (Global)      | 0.6898               | 0.5751              
Inference Time       | 0.1162s (CPU)        | 0.0277        s (GPU)
